# 01 Validate Repository

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


Preflight validation for schema-v2 canonical scientific pairs.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
required=['pyproject.toml','src/wwgpt/analysis.py']+[f'notebooks/{i:02d}_'+name for i,name in [(1,'validate_repository.ipynb'),(2,'compare_single_level.ipynb'),(3,'weightwatcher_analysis.ipynb'),(4,'scaling_laws.ipynb'),(5,'overfitting_and_generalization.ipynb'),(6,'summary_report.ipynb')]]
repo_files=pd.DataFrame([{'path':p,'exists':(repo/p).exists()} for p in required]); repo_files.to_csv(ANALYSIS_DIR/'repository_file_audit.csv',index=False)
notebook_audit=pd.DataFrame([{'notebook':str(p),'valid_nbformat': True} for p in (repo/'notebooks').glob('0*.ipynb')]); notebook_audit.to_csv(ANALYSIS_DIR/'notebook_audit.csv',index=False)
run_rows=[]
for r in runs:
    art=load_run_artifacts(Path(r['run_dir'])); man=art['manifest']; spec=art['spectral']; proj=art['projection']
    run_rows.append({'seed':r['seed'],'pair_id':r['pair_id'],'optimizer_raw':r['optimizer_raw'],'manifest':(Path(r['run_dir'])/'manifest.json').exists(),'metrics':(Path(r['run_dir'])/'metrics.csv').exists(),'spectral':(Path(r['run_dir'])/'spectral.csv').exists(),'complete':(Path(r['run_dir'])/'run_complete.json').exists(),'projection_required_ok': r['optimizer_family']=='adamw' or (Path(r['run_dir'])/'wwpgd_projection.csv').exists(),'scientific_schema_version':man.get('scientific_schema_version'),'spectral_estimator':man.get('spectral_estimator') or (spec['spectral_estimator'].dropna().iloc[0] if not spec.empty and spec['spectral_estimator'].notna().any() else None),'weightwatcher_rows':int((spec.get('spectral_estimator',pd.Series(dtype=str)).astype(str).str.lower()=='weightwatcher').sum())})
run_validity=pd.DataFrame(run_rows); run_validity.to_csv(ANALYSIS_DIR/'run_validity_audit.csv',index=False)
pair_audit.to_csv(ANALYSIS_DIR/'canonical_pair_audit.csv',index=False)
checks=[]
checks.append(('required files', repo_files.exists.all()))
checks.append(('results root exists', RESULTS_ROOT.exists()))
checks.append(('expected seeds', {c.seed for c in canonical_pairs}==EXPECTED_SEEDS))
checks.append(('one pair per seed', len({c.seed for c in canonical_pairs})==len(canonical_pairs)))
checks.append(('two arms per canonical pair', all({'adamw','wwpgd'}.issubset(c.runs) for c in canonical_pairs)))
checks.append(('schema v2', (run_validity.scientific_schema_version.astype(float)>=2).all()))
checks.append(('WeightWatcher spectral rows', (run_validity.weightwatcher_rows>0).all()))
# source SVD regression guard
import inspect, wwgpt.ww as ww
src=inspect.getsource(getattr(ww,'apply_wwpgd_reference', lambda: None))
checks.append(('no redundant post-projection svdvals', 'torch.linalg.svdvals' not in src[src.find('def apply_wwpgd_reference'):]))
summary=pd.DataFrame(checks,columns=['check','passed']); display(summary)
print('## PASS' if summary.passed.all() else '## FAIL')
